# 04 — Mitigación de sesgo y frontera de Pareto

**Tesis:** Estratificación de riesgo clínico consciente de la equidad — sesgo geográfico entre entidades.

Este notebook **corrige** la inequidad detectada en el notebook 03. Cubre:
1. Integración del **índice de marginación de CONAPO 2020** por entidad y su correlación con el sesgo.
2. **Recalibrado** (arreglar la sobreestimación de riesgo).
3. Las tres familias de mitigación: **pre**, **in** y **post**-procesamiento.
4. La **frontera de Pareto** equidad–desempeño (entregable de decisión).
5. Tabla comparativa y conclusiones.

> Hallazgo adelantado y honesto: no todas las técnicas funcionan igual. En este problema, con 32 grupos y subgrupos pequeños, el **post-procesamiento** (umbrales por entidad) es el más efectivo, mientras que el pre e in-procesamiento resultan insuficientes o inestables. Documentar esa comparación es en sí una aportación.


## 1. Datos, modelo e índice de marginación CONAPO 2020

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, roc_curve, brier_score_loss,
                             average_precision_score, balanced_accuracy_score)

import tesis_common as tc

# Cohorte y modelo se cargan del contrato compartido. Este notebook entrenaba
# antes su PROPIO modelo con max_iter=250 y learning_rate=0.06, distintos de los
# del notebook 02 (300 / 0.05): la cadena "modelo de 02 -> auditado en 03 ->
# mitigado en 04" no describía el mismo objeto en las tres etapas.
df, manifiesto = tc.cargar_cohorte()
paquete = tc.cargar_modelo(n_filas_cohorte=len(df))

X = df[tc.PREDICTORES].astype("float32")
y = df[tc.TARGET].astype(int)
ent = df["entidad_nombre"]

idx_tr, idx_cal, idx_te = tc.particionar(y)
Xtr, ytr, etr = X.iloc[idx_tr], y.iloc[idx_tr], ent.iloc[idx_tr]
Xcal, ycal = X.iloc[idx_cal], y.iloc[idx_cal]
Xte, yte, ete = X.iloc[idx_te], y.iloc[idx_te], ent.iloc[idx_te]

ya, ea = yte.to_numpy(), ete.to_numpy()
p_crudo = paquete["modelo"].predict_proba(Xte)[:, 1]   # sin calibrar, solo para contraste
p = tc.puntuar(paquete, Xte)                            # riesgo calibrado: el que se despliega

THR, tpr_obj, fpr_obj = tc.umbral_para_sensibilidad(ya, p)
ents, ents_fuera = tc.grupos_evaluables(ea, ya)

# Índice de marginación por entidad, CONAPO 2020 (archivo IME_2020.xls, con base
# en el Censo INEGI 2020). Escala DP2: MAYOR valor = MENOR marginación. El
# catálogo anterior era un reescalamiento no documentado de estas mismas cifras
# a [0,1], etiquetado como "índice normalizado", nombre que corresponde a las
# ediciones previas y cuya dirección es la contraria.
IM = tc.CONAPO_IM_2020
GRADO = tc.CONAPO_GRADO
if not tc.CONAPO_VERIFICADO:
    print("!! AVISO: indice de marginacion SIN VERIFICAR contra la fuente CONAPO.\n")

print("Modelo:", tc.MODELO_PATH, "| hparams:", paquete["hparams"])
print("Marginacion:", tc.CONAPO_FUENTE)
print("AUROC global:", round(roc_auc_score(ya, p), 4),
      f"| umbral (sens. {tc.SENSIBILIDAD_OBJETIVO:.0%}): {THR:.4f}",
      "| entidades evaluables:", len(ents))
if len(ents_fuera):
    print("Entidades excluidas por soporte insuficiente:")
    print(ents_fuera.to_string(index=False))

Modelo: modelo_hgb_mortalidad.joblib | hparams: {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 6, 'class_weight': 'balanced', 'random_state': 42}
Marginacion: CONAPO, Índices de marginación 2020 (IME_2020.xls), Censo de Población y Vivienda 2020, INEGI
AUROC global: 0.9503 | umbral (sens. 80%): 0.1562 | entidades evaluables: 32


## 2. Funciones auxiliares

In [2]:
# Las tasas y las brechas se calculan con la implementación única de
# tesis_common; aquí solo se envuelven para operar con un mapa de umbrales por
# entidad, que es la forma que toma la mitigación post-procesamiento.
tf = tc.tasas

def gaps(pscore, tmap):
    """Brechas de equidad entre entidades bajo un umbral global o por entidad.

    Args:
        pscore: riesgo calibrado en el bloque de prueba.
        tmap: umbral único (float) o mapa {entidad: umbral}.

    Returns:
        dict con gap_TPR, gap_FPR, tasa de selección global y exactitud balanceada.
    """
    tprs, fprs = [], []
    pred = np.zeros(len(pscore), int)
    for g in ents:
        m = (ea == g)
        t = tmap[g] if isinstance(tmap, dict) else tmap
        a, b = tf(ya[m], pscore[m], t)
        tprs.append(a); fprs.append(b)
        pred[m] = (pscore[m] >= t).astype(int)
    return dict(gap_TPR=max(tprs) - min(tprs), gap_FPR=max(fprs) - min(fprs),
                seleccion=pred.mean(), bal_acc=balanced_accuracy_score(ya, pred))

## 3. ¿La marginación explica el sesgo geográfico?

Se correlaciona el desempeño por entidad (AUROC, TPR) con el **índice de marginación de CONAPO 2020** (archivo `IME_2020.xls`, con base en el Censo INEGI 2020).

> **Escala.** La edición 2020 se calcula por Distancias Ponderadas al cuadrado (DP2) y va de 10.99 (Guerrero, grado «Muy alto») a 23.44 (Nuevo León, «Muy bajo»): un valor **más alto** indica **menor** marginación. El «índice normalizado» acotado a 0–1 pertenece a ediciones anteriores y su dirección es la contraria.

Además de la correlación lineal se agrega el desempeño por **grado** de marginación, que es la forma en que CONAPO comunica el índice, con una prueba de Kruskal–Wallis entre grados.


In [3]:
from scipy import stats

rows = [(g, roc_auc_score(ya[ea == g], p[ea == g]),
         tf(ya[ea == g], p[ea == g], THR)[0], IM[g], GRADO[g]) for g in ents]
Rm = pd.DataFrame(rows, columns=["entidad", "AUROC", "TPR", "marginacion_IM2020", "grado"])

# Con 32 puntos hay que reportar significancia: la conclusión "el sesgo no se
# explica por marginacion" es una afirmacion de la discusion, no un r suelto.
CORR_MARGINACION = {}
for col in ["AUROC", "TPR"]:
    r, pr = stats.pearsonr(Rm[col], Rm["marginacion_IM2020"])
    rho, ps = stats.spearmanr(Rm[col], Rm["marginacion_IM2020"])
    CORR_MARGINACION[col.lower()] = {"r": round(float(r), 3), "p_r": round(float(pr), 3),
                                     "rho": round(float(rho), 3), "p_rho": round(float(ps), 3),
                                     "n": int(len(Rm))}
    print(f"{col:6s} vs IM_2020:  r={r:+.3f} (p={pr:.3f})   rho={rho:+.3f} (p={ps:.3f})   n={len(Rm)}")

# Agregado por grado de marginación: una correlación lineal puede ocultar un
# patrón escalonado, y el grado es la forma en que CONAPO comunica el índice.
POR_GRADO = (Rm.groupby("grado")
               .agg(n_entidades=("entidad", "size"),
                    AUROC_medio=("AUROC", "mean"),
                    TPR_medio=("TPR", "mean"))
               .reindex([g for g in tc.GRADOS_ORDEN if g in set(Rm["grado"])])
               .round(4))
print("\n== Desempeno medio por grado de marginacion (de mayor a menor carencia) ==")
print(POR_GRADO.to_string())

# Kruskal-Wallis: ¿difiere el AUROC entre grados? Es la prueba adecuada para
# comparar mas de dos grupos ordinales sin suponer normalidad.
grupos_auroc = [g["AUROC"].to_numpy() for _, g in Rm.groupby("grado")]
H, p_kw = stats.kruskal(*grupos_auroc)
print(f"\nKruskal-Wallis AUROC entre grados: H={H:.3f}, p={p_kw:.3f}")

print("\n(IM_2020 alto = MENOR marginacion). Correlaciones debiles y no")
print("significativas indican que el sesgo NO se explica solo por marginacion:")
print("intervienen tamano de muestra, mezcla de casos y calidad del dato.")

AUROC  vs IM_2020:  r=-0.071 (p=0.700)   rho=-0.075 (p=0.684)   n=32
TPR    vs IM_2020:  r=-0.178 (p=0.328)   rho=-0.268 (p=0.138)   n=32

== Desempeno medio por grado de marginacion (de mayor a menor carencia) ==
          n_entidades  AUROC_medio  TPR_medio
grado                                        
Muy alto            3       0.9485     0.7932
Alto                9       0.9451     0.8084
Medio               8       0.9484     0.8162
Bajo                8       0.9486     0.7936
Muy bajo            4       0.9399     0.7093

Kruskal-Wallis AUROC entre grados: H=1.827, p=0.768

(IM_2020 alto = MENOR marginacion). Correlaciones debiles y no
significativas indican que el sesgo NO se explica solo por marginacion:
intervienen tamano de muestra, mezcla de casos y calidad del dato.


## 4. Verificación del recalibrado (isotónico)

El modelo con `class_weight='balanced'` sobreestima el riesgo absoluto. La regresión isotónica que lo corrige se ajusta en el **notebook 02**, sobre un bloque de calibración separado del de prueba, y todos los notebooks consumen el riesgo calibrado vía `tesis_common.puntuar`.

Aquí solo se verifica su efecto sobre el bloque de prueba íntegro. La versión anterior ajustaba el calibrador en esta celda usando la mitad del conjunto de prueba y reportaba sobre la otra mitad, mientras las secciones 5 a 8 evaluaban sobre el test completo —incluidas las filas de calibración—, lo que contaminaba la evaluación.


In [4]:
# La calibración se ajusta en el notebook 02, sobre un bloque de calibración
# propio. Antes se ajustaba AQUÍ con la mitad del conjunto de prueba y se
# reportaba sobre la otra mitad, mientras las secciones 5-8 usaban el test
# completo: la evaluación quedaba contaminada con los datos de calibración.
# Aquí solo se verifica el efecto sobre el bloque de prueba íntegro.
print(f"Brier                  crudo -> calibrado:  "
      f"{brier_score_loss(ya, p_crudo):.4f}  ->  {brier_score_loss(ya, p):.4f}")
print(f"Riesgo medio predicho  crudo -> calibrado:  "
      f"{p_crudo.mean():.3f}  ->  {p.mean():.3f}   (observado {ya.mean():.3f})")
print(f"AUROC (invariante: la isotónica es monótona): "
      f"{roc_auc_score(ya, p_crudo):.4f}  ->  {roc_auc_score(ya, p):.4f}")
print(f"AUPRC (invariante):                          "
      f"{average_precision_score(ya, p_crudo):.4f}  ->  {average_precision_score(ya, p):.4f}")

Brier                  crudo -> calibrado:  0.0866  ->  0.0323
Riesgo medio predicho  crudo -> calibrado:  0.205  ->  0.056   (observado 0.056)
AUROC (invariante: la isotónica es monótona): 0.9504  ->  0.9503
AUPRC (invariante):                          0.5594  ->  0.5533


## 5. Mitigación PRE-procesamiento (reponderación por entidad)

In [5]:
# Reponderación inversa al tamaño de cada entidad: más peso a los estados con
# menos registros. Reentrenar es inherente al pre-procesamiento, pero se usan los
# hiperparámetros CANÓNICOS (tc.HPARAMS_HGB) para que la única diferencia frente
# al baseline sean los pesos y la comparación sea limpia.
cnt = etr.value_counts()
etr_a = etr.to_numpy()
w = np.ones(len(etr))
for g, c in cnt.items():
    w[etr_a == g] = len(etr) / (len(cnt) * c)

model_pre = HistGradientBoostingClassifier(**tc.HPARAMS_HGB).fit(Xtr, ytr, sample_weight=w)

# Se calibra en el MISMO bloque de calibración que el baseline: así la tasa de
# selección y los umbrales de ambos métodos viven en la misma escala.
cal_pre = IsotonicRegression(out_of_bounds="clip").fit(
    model_pre.predict_proba(Xcal)[:, 1], ycal)
p_pre = cal_pre.predict(model_pre.predict_proba(Xte)[:, 1])

THR_pre, _, _ = tc.umbral_para_sensibilidad(ya, p_pre)
res_pre = gaps(p_pre, THR_pre)
print("PRE:", {k: round(float(v), 3) for k, v in res_pre.items()},
      "| AUROC", round(roc_auc_score(ya, p_pre), 4),
      "| AUPRC", round(average_precision_score(ya, p_pre), 3))
print(" -> la reponderacion apenas cambia la brecha: insuficiente por si sola.")

PRE: {'gap_TPR': 0.29, 'gap_FPR': 0.082, 'seleccion': 0.099, 'bal_acc': 0.865} | AUROC 0.9501 | AUPRC 0.554
 -> la reponderacion apenas cambia la brecha: insuficiente por si sola.


## 6. Mitigación IN-procesamiento (Fairlearn, EqualizedOdds)

Reducción con restricción de equalized odds sobre una submuestra (el método reentrena muchas veces). Se documenta su comportamiento.


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

# La submuestra se toma del bloque de ENTRENAMIENTO. Antes se muestreaba de `df`
# completo, con lo que entraban filas del conjunto de prueba del baseline y el
# resultado no era comparable con ninguno de los otros métodos.
N_SUB = 120_000
sub = np.random.RandomState(3).choice(idx_tr, size=min(N_SUB, len(idx_tr)), replace=False)

Xs = X.iloc[sub].copy()
Xs = Xs.fillna(Xs.median())          # ExponentiatedGradient + LR no admite NaN
ys = y.iloc[sub]
es = ent.iloc[sub].astype(str)

Xstr, Xste, ystr, yste, estr, este = train_test_split(
    Xs, ys, es, test_size=0.3, stratify=ys, random_state=tc.SEMILLA)
sc = StandardScaler().fit(Xstr)

eg = ExponentiatedGradient(
    LogisticRegression(max_iter=500, class_weight="balanced", random_state=tc.SEMILLA),
    constraints=EqualizedOdds(), max_iter=20)
eg.fit(sc.transform(Xstr), ystr, sensitive_features=estr)
# predict() muestrea de la distribucion sobre clasificadores que devuelve la
# reduccion: sin random_state el resultado NO es reproducible entre corridas.
pred_in = eg.predict(sc.transform(Xste), random_state=tc.SEMILLA)

ys_a, es_a = yste.to_numpy(), este.to_numpy()
ents_s, _ = tc.grupos_evaluables(es_a, ys_a, min_n=200, min_pos=10)

def tpr_only(yt, pr):
    """Sensibilidad a partir de predicciones binarias (no de puntajes)."""
    tp = int(((pr == 1) & (yt == 1)).sum())
    fn = int(((pr == 0) & (yt == 1)).sum())
    return tp / (tp + fn) if (tp + fn) else np.nan

def gap_tpr(pred):
    """Brecha maxima de sensibilidad entre entidades evaluables."""
    t = [tpr_only(ys_a[es_a == g], pred[es_a == g]) for g in ents_s]
    return float(np.nanmax(t) - np.nanmin(t))

gap_in = gap_tpr(pred_in)

# Estabilidad del metodo: la reduccion devuelve una DISTRIBUCION sobre
# clasificadores, asi que cada realizacion da una brecha distinta AUNQUE el
# modelo ajustado sea el mismo. Se cuantifica ese rango en lugar de reportar un
# unico numero, porque la dispersion es el resultado relevante.
GAPS_IN = [gap_tpr(eg.predict(sc.transform(Xste), random_state=s)) for s in range(20)]
gap_in_min, gap_in_max = float(np.min(GAPS_IN)), float(np.max(GAPS_IN))
gap_in_mediana = float(np.median(GAPS_IN))

print(f"IN (Fairlearn EqualizedOdds, submuestra de entrenamiento n={len(Xs):,}): "
      f"TPR-gap = {gap_in:.3f} sobre {len(ents_s)} entidades (semilla {tc.SEMILLA})")
print(f" -> sobre 20 realizaciones del mismo modelo ajustado: mediana {gap_in_mediana:.3f}, "
      f"rango [{gap_in_min:.3f}, {gap_in_max:.3f}]")
print(" -> la brecha depende mas de la realizacion que de la restriccion: con 32 grupos")
print("    y subgrupos pequenos el metodo es inestable y no reduce la disparidad.")
print(" -> evaluado en submuestra y con modelo lineal: no comparable 1:1 con el baseline.")

IN (Fairlearn EqualizedOdds, submuestra de entrenamiento n=120,000): TPR-gap = 0.445 sobre 32 entidades (semilla 42)
 -> sobre 20 realizaciones del mismo modelo ajustado: mediana 0.397, rango [0.245, 0.628]
 -> la brecha depende mas de la realizacion que de la restriccion: con 32 grupos
    y subgrupos pequenos el metodo es inestable y no reduce la disparidad.
 -> evaluado en submuestra y con modelo lineal: no comparable 1:1 con el baseline.


## 7. Mitigación POST-procesamiento (umbrales por entidad)

Se elige un umbral por entidad para igualar la sensibilidad objetivo (80 %) en cada una.

Los umbrales se estiman en el **bloque de calibración** y se evalúan en el de prueba. La versión anterior los derivaba del propio conjunto de prueba, lo que producía una brecha residual optimista (ajuste en muestra) que no se sostendría en despliegue; ambas cifras se reportan para dimensionar esa diferencia.


In [7]:
# Un umbral por entidad, elegido para alcanzar la sensibilidad objetivo DENTRO de
# cada estado. Los umbrales se estiman en el bloque de CALIBRACIÓN y se evalúan en
# el de prueba: derivarlos del propio test (como hacía la versión anterior) da una
# brecha residual optimista que no se sostendría en despliegue.
p_cal = tc.puntuar(paquete, Xcal)
e_cal = ent.iloc[idx_cal].to_numpy()
y_cal_a = ycal.to_numpy()

tmap = {}
sin_soporte = []
for g in ents:
    m = (e_cal == g)
    if m.sum() >= tc.MIN_N_GRUPO and y_cal_a[m].sum() >= tc.MIN_POS_GRUPO:
        tmap[g] = tc.umbral_para_sensibilidad(y_cal_a[m], p_cal[m])[0]
    else:
        tmap[g] = THR          # sin soporte para estimar: se conserva el global
        sin_soporte.append(g)

# Contraste: los mismos umbrales estimados en test (en muestra, optimista).
tmap_insample = {g: tc.umbral_para_sensibilidad(ya[ea == g], p[ea == g])[0] for g in ents}

res_base = gaps(p, THR)
res_post = gaps(p, tmap)
res_post_opt = gaps(p, tmap_insample)

print("BASELINE (umbral global)      :", {k: round(float(v), 3) for k, v in res_base.items()})
print("POST (umbral/entidad, honesto):", {k: round(float(v), 3) for k, v in res_post.items()})
print("POST (umbral/entidad, en test):", {k: round(float(v), 3) for k, v in res_post_opt.items()})
print(f"\n -> la brecha de TPR cae de {res_base['gap_TPR']:.3f} a {res_post['gap_TPR']:.3f} "
      f"con umbrales estimados fuera de muestra (cota optimista: {res_post_opt['gap_TPR']:.3f}).")
if sin_soporte:
    print(" -> entidades sin soporte en calibracion (conservan umbral global):", sin_soporte)

BASELINE (umbral global)      : {'gap_TPR': 0.294, 'gap_FPR': 0.087, 'seleccion': 0.1, 'bal_acc': 0.868}
POST (umbral/entidad, honesto): {'gap_TPR': 0.14, 'gap_FPR': 0.101, 'seleccion': 0.102, 'bal_acc': 0.871}
POST (umbral/entidad, en test): {'gap_TPR': 0.022, 'gap_FPR': 0.107, 'seleccion': 0.101, 'bal_acc': 0.869}

 -> la brecha de TPR cae de 0.294 a 0.140 con umbrales estimados fuera de muestra (cota optimista: 0.022).


## 8. Frontera de Pareto equidad–desempeño

Se interpola el umbral entre el global (máximo desempeño, máxima brecha) y el por-entidad (mínima brecha). Cada punto es una política de despliegue posible; el entregable para un regulador es esta curva.


In [8]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

# Se interpola entre el umbral global (alfa=0) y el umbral por entidad estimado
# fuera de muestra (alfa=1). Cada alfa es una política de despliegue posible.
alphas = np.linspace(0, 1, 11)
frontera = []
for a in alphas:
    tm = {g: (1 - a) * THR + a * tmap[g] for g in ents}
    r = gaps(p, tm)
    frontera.append((a, r["gap_TPR"], r["bal_acc"], r["seleccion"]))
F = pd.DataFrame(frontera, columns=["alpha", "gap_TPR", "bal_acc", "seleccion"])
print(F.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].plot(F["gap_TPR"], F["bal_acc"], "o-", color="#1F3864")
for _, r in F.iterrows():
    ax[0].annotate(f"{r['alpha']:.1f}", (r["gap_TPR"], r["bal_acc"]), fontsize=8)
ax[0].set_xlabel("Brecha de TPR entre entidades (menor = mas justo)")
ax[0].set_ylabel("Exactitud balanceada (desempeno)")
ax[0].set_title("Frontera de Pareto equidad-desempeno")
ax[0].invert_xaxis()

ax[1].scatter(Rm["marginacion_IM2020"], Rm["AUROC"], color="#2E5496")
ax[1].set_xlabel("Indice de marginacion CONAPO 2020 (DP2; alto = menos marginado)")
ax[1].set_ylabel("AUROC por entidad")
titulo = "Desempeno vs marginacion por entidad"
if not tc.CONAPO_VERIFICADO:
    titulo += "\n(indice PROVISIONAL, sin verificar)"
ax[1].set_title(titulo)
for _, r in Rm.iterrows():
    if r["grado"] in ("Muy alto", "Muy bajo"):
        ax[1].annotate(r["entidad"].title(), (r["marginacion_IM2020"], r["AUROC"]),
                       fontsize=7, xytext=(3, 3), textcoords="offset points")

plt.tight_layout(); plt.savefig("pareto_mitigacion.png", dpi=110)
print("\nFigura guardada: pareto_mitigacion.png")

 alpha  gap_TPR  bal_acc  seleccion
   0.0    0.294    0.868      0.100
   0.1    0.281    0.867      0.099
   0.2    0.275    0.866      0.098
   0.3    0.249    0.867      0.098
   0.4    0.234    0.869      0.099
   0.5    0.221    0.871      0.100
   0.6    0.221    0.871      0.100
   0.7    0.218    0.870      0.099
   0.8    0.218    0.870      0.100
   0.9    0.184    0.869      0.100
   1.0    0.140    0.871      0.102

Figura guardada: pareto_mitigacion.png


## 9. Tabla comparativa de intervenciones

In [9]:
import json

tabla = pd.DataFrame([
    {"metodo": "Baseline (umbral global)",       **{k: round(float(v), 3) for k, v in res_base.items()}},
    {"metodo": "PRE (reponderacion entidad)",    **{k: round(float(v), 3) for k, v in res_pre.items()}},
    {"metodo": "IN (Fairlearn EqOdds)*",         "gap_TPR": round(gap_in, 3),
     "gap_FPR": np.nan, "seleccion": np.nan, "bal_acc": np.nan},
    {"metodo": "POST (umbral/entidad)",          **{k: round(float(v), 3) for k, v in res_post.items()}},
    {"metodo": "POST (umbral/entidad, en test)", **{k: round(float(v), 3) for k, v in res_post_opt.items()}},
])
print(tabla.to_string(index=False))
print("\n* IN evaluado en submuestra de entrenamiento y con modelo lineal; no comparable 1:1.")
print("  La ultima fila ajusta umbrales en el propio test: cota optimista, no desplegable.")

# Los resultados se exportan para que el documento LaTeX cite exactamente estas
# cifras en lugar de transcribirlas a mano, que es como se introducen las
# inconsistencias entre el capitulo de resultados y los notebooks.
resultados = {
    "cohorte": {"n_filas": int(len(df)), "solo_confirmados": manifiesto["solo_confirmados"],
                "tasa_defuncion": manifiesto["tasa_defuncion"]},
    "modelo": {"hparams": paquete["hparams"],
               "auroc_global": round(float(roc_auc_score(ya, p)), 4),
               "auprc_global": round(float(average_precision_score(ya, p)), 4),
               "brier_crudo": round(float(brier_score_loss(ya, p_crudo)), 4),
               "brier_calibrado": round(float(brier_score_loss(ya, p)), 4),
               "umbral": round(float(THR), 4),
               "sensibilidad_objetivo": tc.SENSIBILIDAD_OBJETIVO},
    "mitigacion": tabla.replace({np.nan: None}).to_dict(orient="records"),
    "frontera_pareto": F.round(4).to_dict(orient="records"),
    "corr_marginacion": CORR_MARGINACION,
    "conapo_fuente": tc.CONAPO_FUENTE,
    "marginacion_por_grado": POR_GRADO.reset_index().to_dict(orient="records"),
    "in_estabilidad": {"mediana": round(gap_in_mediana, 3),
                       "min": round(gap_in_min, 3), "max": round(gap_in_max, 3),
                       "n_realizaciones": len(GAPS_IN)},
    "kruskal_auroc_grado": {"H": round(float(H), 3), "p": round(float(p_kw), 3)},
    "conapo_verificado": tc.CONAPO_VERIFICADO,
}
with open("resultados_tesis.json", "w", encoding="utf-8") as fh:
    json.dump(resultados, fh, indent=2, ensure_ascii=False)
print("\nResultados exportados: resultados_tesis.json")

                        metodo  gap_TPR  gap_FPR  seleccion  bal_acc
      Baseline (umbral global)    0.294    0.087      0.100    0.868
   PRE (reponderacion entidad)    0.290    0.082      0.099    0.865
        IN (Fairlearn EqOdds)*    0.445      NaN        NaN      NaN
         POST (umbral/entidad)    0.140    0.101      0.102    0.871
POST (umbral/entidad, en test)    0.022    0.107      0.101    0.869

* IN evaluado en submuestra de entrenamiento y con modelo lineal; no comparable 1:1.
  La ultima fila ajusta umbrales en el propio test: cota optimista, no desplegable.



Resultados exportados: resultados_tesis.json


## 10. Conclusiones y próximos pasos (notebook 05)

**Resultados de la mitigación:**
- El **recalibrado** corrige la sobreestimación de riesgo (Brier 0.087 → 0.032) sin tocar el AUROC.
- El **pre-procesamiento** por sí solo casi no reduce el sesgo geográfico (0.294 → 0.290).
- El **in-procesamiento** (reducción con equalized odds) es inestable: sobre el mismo modelo ajustado, 20 realizaciones dan brechas entre 0.245 y 0.628. El rango contiene a la línea de base, así que el método no permite concluir nada en este régimen.
- El **post-procesamiento** (umbrales por entidad, estimados en calibración) reduce la brecha de 0.294 a 0.140 sin costo de desempeño agregado. No la elimina.
- La **frontera de Pareto** muestra que la equidad se puede mejorar sin sacrificar desempeño global, pero el descenso no es proporcional: se estanca entre α = 0.5 y α = 0.8.
- La **marginación de CONAPO no explica el sesgo**: ni por correlación (r = −0.07, p = 0.70) ni al agregar por grado (Kruskal–Wallis p = 0.77). H2 queda refutada tal como se operacionalizó.

**Notebook 05 — validación externa:** repetir modelo + auditoría + mitigación sobre la base de Egresos Hospitalarios de la DGIS para evaluar la transportabilidad del desempeño y de la equidad.
